In [2]:
"""
Module 3 — Heteroscedastic GP Model
=====================================
Builds and trains two independent heteroscedastic Sparse Variational
Gaussian Process (SVGP) models — one for log_k0, one for alpha.

Why heteroscedastic?
    The original paper used a homoscedastic GP which assumes noise is
    constant across the composition space. This caused overfitting of
    log_k0 because the log transform compressed the dynamic range,
    making the noise appear artificially small everywhere.

    A heteroscedastic GP uses a SECOND latent GP to learn how noise
    varies across composition space — regions where LSV fits were
    unreliable get higher noise, preventing overfitting.

Why SVGP (Sparse Variational GP)?
    Exact GP inference scales as O(n³) — 966 points is manageable but
    the active learning loop retrains many times. SVGP uses m inducing
    points (m << n) to approximate the full GP at O(m²n) cost, making
    repeated retraining fast.

Architecture per output (log_k0 and alpha each get their own model):
    ┌─────────────────────────────────────────────────┐
    │  Input: [Au%, Ir%]  (2D composition space)      │
    │                                                  │
    │  GP₁ (signal kernel)  → predicts mean f(x)      │
    │  GP₂ (noise kernel)   → predicts log σ²(x)      │
    │                                                  │
    │  Likelihood: HeteroskedasticTFPConditional       │
    │  (Normal distribution with learned scale)        │
    └─────────────────────────────────────────────────┘

Input:
    - seed_set.csv    (output of Module 2, initial training data)
    - pool_set.csv    (output of Module 2, points to predict)

Output:
    - Trained model objects (in memory, passed to Module 4)
    - predictions.csv (mean + variance for all pool points)
"""

'\nModule 3 — Heteroscedastic GP Model\n=====================================\nBuilds and trains two independent heteroscedastic Sparse Variational\nGaussian Process (SVGP) models — one for log_k0, one for alpha.\n\nWhy heteroscedastic?\n    The original paper used a homoscedastic GP which assumes noise is\n    constant across the composition space. This caused overfitting of\n    log_k0 because the log transform compressed the dynamic range,\n    making the noise appear artificially small everywhere.\n\n    A heteroscedastic GP uses a SECOND latent GP to learn how noise\n    varies across composition space — regions where LSV fits were\n    unreliable get higher noise, preventing overfitting.\n\nWhy SVGP (Sparse Variational GP)?\n    Exact GP inference scales as O(n³) — 966 points is manageable but\n    the active learning loop retrains many times. SVGP uses m inducing\n    points (m << n) to approximate the full GP at O(m²n) cost, making\n    repeated retraining fast.\n\nArchitecture

In [3]:
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import gpflow
from gpflow.utilities import print_summary
from pathlib import Path

# Suppress TensorFlow and GPflow verbose logs
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

In [8]:
# GP input columns — composition space (2D after dropping Rh)
INPUT_COLS  = ['Au [at.%]', 'Ir [at.%]']

# GP output columns — one model per output
OUTPUT_COLS = ['log_k0', 'alpha']

# ─────────────────────────────────────────────────────────────────────────────
# INPUT NORMALISATION
# ─────────────────────────────────────────────────────────────────────────────

class Normaliser:
    """
    Normalises GP inputs to zero mean and unit variance.

    Why normalise?
    --------------
    GP kernel lengthscales are initialised at 1.0 by default.
    If Au% ranges 20-80 and Ir% ranges 5-70, the raw scales are very
    different. Normalising puts both inputs on the same scale so the
    kernel can initialise sensibly and converge faster.

    The normaliser is fit on the SEED SET only (not the full dataset)
    to simulate the real active learning scenario where we don't know
    the full data distribution upfront.
    """
    def __init__(self):
        self.mean_ = None
        self.std_  = None

    def fit(self, X: np.ndarray) -> 'Normaliser':
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8   # add epsilon to avoid div by zero nd crashing the gp
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X - self.mean_) / self.std_

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)
 
# ─────────────────────────────────────────────────────────────────────────────
# D_MAX COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────
from scipy.spatial.distance import pdist
def compute_d_max(seed_df: pd.DataFrame, pool_df: pd.DataFrame) -> float:
    """
    Compute the maximum Euclidean distance in composition space
    across ALL 966 points (seed + pool).
 
    This is the normalisation constant for the within-library
    movement penalty. By dividing all within-library distances
    by d_max, we ensure they stay in [0, 1] — always below
    the switch-library penalty of 1 + gamma.
 
    Parameters
    ----------
    seed_df : seed set DataFrame
    pool_df : pool set DataFrame
 
    Returns
    -------
    d_max : float — maximum pairwise composition distance
    """
    all_points = pd.concat([seed_df, pool_df])[INPUT_COLS].values
    d_max = pdist(all_points).max()
    print(f"  d_max (max composition distance): {d_max:.4f} at.% units")
    return float(d_max)
    
# ─────────────────────────────────────────────────────────────────────────────
# MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_heteroscedastic_svgp(
    X_init      : np.ndarray,
    n_inducing  : int = 20,
) -> gpflow.models.SVGP:
    """
    Build a heteroscedastic SVGP model.

    The model has TWO latent GPs:
        GP₁ — Matern32 kernel → models the signal (mean prediction)
        GP₂ — Matern32 kernel → models log noise variance σ²(x)

    Both GPs share the same input space [Au%, Ir%] but have independent
    kernels and inducing points, allowing them to learn at different
    length scales.

    The HeteroskedasticTFPConditional likelihood combines them:
        y ~ Normal(GP₁(x),  exp(GP₂(x)))
                   ↑signal      ↑learned noise

    Parameters
    ----------
    X_init     : initial training inputs, used to initialise inducing points
    n_inducing : number of inducing points (default 20)
                 More = better approximation but slower training
                 With 24 seed points, 20 inducing points is near-exact

    Returns
    -------
    gpflow.models.SVGP — untrained model ready for optimisation
    """
    # ── Kernels ───────────────────────────────────────────────────────────────
    # Matern 3/2 is a good default for physical properties —

    signal_kernel = gpflow.kernels.Matern32(
        active_dims=[0, 1],          # use both input dimensions
        lengthscales=[1.0, 1.0],     # one lengthscale per input dimension
    )
    noise_kernel = gpflow.kernels.Matern32(
        active_dims=[0, 1],
        lengthscales=[1.0, 1.0],
    )

    # SeparateIndependent wraps both kernels — each latent GP gets its own
    kernel = gpflow.kernels.SeparateIndependent([signal_kernel, noise_kernel])

    # ── Likelihood ────────────────────────────────────────────────────────────
    # HeteroskedasticTFPConditional:
    #   - Takes 2 latent function values [f₁, f₂]
    #   - f₁ = predicted mean (signal GP output)
    #   - f₂ = log noise scale (noise GP output, exponentiated to ensure > 0)
    #   - Returns Normal(f₁, exp(f₂))
    likelihood = gpflow.likelihoods.HeteroskedasticTFPConditional(
        distribution_class=tfp.distributions.Normal,
        scale_transform=tfp.bijectors.Exp(),   # ensures σ > 0
    )

    # ── Inducing points ───────────────────────────────────────────────────────
    # Initialise inducing points spread across the training data
    # Using k-means-like spread: pick n_inducing evenly spaced points
    n_inducing = min(n_inducing, len(X_init))
    indices    = np.linspace(0, len(X_init) - 1, n_inducing, dtype=int)
    Z_init     = X_init[indices].copy()

    # Each latent GP gets its own set of inducing points
    inducing_variable = gpflow.inducing_variables.SeparateIndependentInducingVariables([
        gpflow.inducing_variables.InducingPoints(Z_init.copy()),
        gpflow.inducing_variables.InducingPoints(Z_init.copy()),
    ])

    # ── Assemble SVGP ─────────────────────────────────────────────────────────
    model = gpflow.models.SVGP(
        kernel            = kernel,
        likelihood        = likelihood,
        inducing_variable = inducing_variable,
        num_latent_gps    = likelihood.latent_dim,   # = 2
    )

    return model

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def train_model(
    model      : gpflow.models.SVGP,
    X_train    : np.ndarray,
    Y_train    : np.ndarray,
    max_iter   : int = 500,
    verbose    : bool = False,
) -> dict:
    """
    Train a heteroscedastic SVGP using Scipy L-BFGS-B optimiser.

    The optimiser maximises the Evidence Lower BOund (ELBO) —
    a lower bound on the log marginal likelihood that accounts for
    the variational approximation introduced by the inducing points.

    Parameters
    ----------
    model     : built (untrained) SVGP model
    X_train   : normalised input array, shape (n, 2)
    Y_train   : output array, shape (n, 1)
    max_iter  : maximum optimisation iterations (default 500)
    verbose   : print ELBO at start and end

    Returns
    -------
    dict with training history: {'elbo_initial', 'elbo_final'}
    """
    data = (
        tf.constant(X_train, dtype=tf.float64),
        tf.constant(Y_train, dtype=tf.float64),
    )

    # Compute ELBO before training
    elbo_initial = model.elbo(data).numpy()

    if verbose:
        print(f"    ELBO before training: {elbo_initial:.4f}")

    # Scipy L-BFGS-B — efficient second-order optimiser
    # Better than Adam for this problem size (< 1000 points)
    opt = gpflow.optimizers.Scipy()
    opt.minimize(
        model.training_loss_closure(data),
        model.trainable_variables,
        options={'maxiter': max_iter, 'disp': False},
    )

    elbo_final = model.elbo(data).numpy()

    if verbose:
        print(f"    ELBO after training:  {elbo_final:.4f}")

    return {'elbo_initial': elbo_initial, 'elbo_final': elbo_final}

# ─────────────────────────────────────────────────────────────────────────────
# PREDICTION
# ─────────────────────────────────────────────────────────────────────────────

def predict(
    model      : gpflow.models.SVGP,
    X_test     : np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Predict mean and variance for test points.

    predict_y() returns the POSTERIOR PREDICTIVE distribution —
    this includes both:
        - epistemic uncertainty (GP uncertainty about the function)
        - aleatoric uncertainty (learned noise from GP₂)

    For the acquisition function in active learning we use the
    TOTAL variance (epistemic + aleatoric) to find where the model
    is most uncertain.

    Parameters
    ----------
    model  : trained SVGP
    X_test : normalised test inputs, shape (n, 2)

    Returns
    -------
    mu  : predicted mean,     shape (n,)
    var : predicted variance, shape (n,)
    """
    X_tf = tf.constant(X_test, dtype=tf.float64)
    mu, var = model.predict_y(X_tf)
    return mu.numpy().flatten(), var.numpy().flatten()


# ─────────────────────────────────────────────────────────────────────────────
# WRAPPER — builds, trains, predicts for one output
# ─────────────────────────────────────────────────────────────────────────────

def fit_and_predict(
    X_train    : np.ndarray,
    Y_train    : np.ndarray,
    X_test     : np.ndarray,
    output_name: str,
    n_inducing : int  = 20,
    max_iter   : int  = 500,
    verbose    : bool = True,
) -> tuple[gpflow.models.SVGP, np.ndarray, np.ndarray]:
    """
    Build, train, and predict for a single output (log_k0 or alpha).

    Parameters
    ----------
    X_train     : training inputs,  shape (n_train, 2)
    Y_train     : training outputs, shape (n_train, 1)
    X_test      : test inputs,      shape (n_test,  2)
    output_name : 'log_k0' or 'alpha' — used for print labels
    n_inducing  : number of inducing points
    max_iter    : optimisation iterations
    verbose     : print progress

    Returns
    -------
    model : trained SVGP
    mu    : predicted means,     shape (n_test,)
    var   : predicted variances, shape (n_test,)
    """
    if verbose:
        print(f"\n  ── Training model for: {output_name} ──────────────────")
        print(f"     Training points : {len(X_train)}")
        print(f"     Inducing points : {n_inducing}")
        print(f"     Max iterations  : {max_iter}")

    model  = build_heteroscedastic_svgp(X_train, n_inducing=n_inducing)
    history = train_model(model, X_train, Y_train, max_iter=max_iter, verbose=verbose)
    mu, var = predict(model, X_test)

    if verbose:
        print(f"     ELBO improvement: {history['elbo_final'] - history['elbo_initial']:.4f}")

    return model, mu, var

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN — train both models and save predictions
# ─────────────────────────────────────────────────────────────────────────────

def run_gpr(
    seed_path  : str,
    pool_path  : str,
    output_dir : str  = "data",
    n_inducing : int  = 20,
    max_iter   : int  = 500,
) -> tuple[dict, pd.DataFrame]:
    """
    Train heteroscedastic GP models for log_k0 and alpha.
    Predict on pool set and save results.

    Parameters
    ----------
    seed_path  : path to seed_set.csv
    pool_path  : path to pool_set.csv
    output_dir : where to save predictions.csv
    n_inducing : number of SVGP inducing points
    max_iter   : optimisation iterations per model

    Returns
    -------
    models : dict {'log_k0': model, 'alpha': model}
    preds  : DataFrame with pool predictions + uncertainties
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Load data ─────────────────────────────────────────────────────────────
    print("=" * 55)
    print("STEP 1: Loading seed and pool sets")
    print("=" * 55)
    seed_df = pd.read_csv(seed_path)
    pool_df = pd.read_csv(pool_path)
    print(f"  Seed set : {len(seed_df)} points")
    print(f"  Pool set : {len(pool_df)} points")

    # ── Extract arrays ────────────────────────────────────────────────────────
    X_seed = seed_df[INPUT_COLS].values.astype(np.float64)
    X_pool = pool_df[INPUT_COLS].values.astype(np.float64)

    # ── d_max ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("STEP 2: Computing composition space diameter (d_max)")
    print("=" * 55)
    d_max = compute_d_max(seed_df, pool_df)
 
    # ── Normalise inputs ──────────────────────────────────────────────────────
    # Fit normaliser on seed set only
    print("\n" + "=" * 55)
    print("STEP 2: Normalising inputs")
    print("=" * 55)
    normaliser = Normaliser()
    X_seed_norm = normaliser.fit_transform(X_seed)
    X_pool_norm = normaliser.transform(X_pool)
    print(f"  Input mean : {normaliser.mean_}")
    print(f"  Input std  : {normaliser.std_}")

    # ── Train models ──────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("STEP 3: Training heteroscedastic GP models")
    print("=" * 55)
    models = {}
    predictions = {}

    for output_col in OUTPUT_COLS:
        Y_seed = seed_df[output_col].values.reshape(-1, 1).astype(np.float64)

        model, mu, var = fit_and_predict(
            X_train     = X_seed_norm,
            Y_train     = Y_seed,
            X_test      = X_pool_norm,
            output_name = output_col,
            n_inducing  = n_inducing,
            max_iter    = max_iter,
            verbose     = True,
        )

        models[output_col] = model
        predictions[f'{output_col}_mu']  = mu
        predictions[f'{output_col}_var'] = var

     # ── Assemble ──────────────────────────────────────────────────────────────
    preds_df = pool_df[['library', 'area',
                         'Au [at.%]', 'Ir [at.%]',
                         'log_k0', 'alpha']].copy()
 
    preds_df['log_k0_mu']  = predictions['log_k0_mu']
    preds_df['log_k0_var'] = predictions['log_k0_var']
    preds_df['alpha_mu']   = predictions['alpha_mu']
    preds_df['alpha_var']  = predictions['alpha_var']
 
    # Save raw predictions (no acquisition score — computed in Module 4)
    pred_path = output_dir / "predictions.csv"
    preds_df.to_csv(pred_path, index=False)
 
    print("\n" + "=" * 55)
    print("DONE")
    print("=" * 55)
    print(f"  Predictions saved to : {pred_path}")
    print(f"  log_k0 var range     : {predictions['log_k0_var'].min():.4f} → {predictions['log_k0_var'].max():.4f}")
    print(f"  alpha  var range     : {predictions['alpha_var'].min():.4f} → {predictions['alpha_var'].max():.4f}")
 
    return models, preds_df, normaliser, d_max
# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("\n" + "=" * 55)
    print("MODULE 3 — HETEROSCEDASTIC GP MODEL")
    print("=" * 55)

    seed_path = input("\nPath to seed_set.csv:\n> ").strip().strip('"').strip("'")
    pool_path = input("\nPath to pool_set.csv:\n> ").strip().strip('"').strip("'")

    n_ind_input = input("\nNumber of inducing points [default: 20]:\n> ").strip()
    n_inducing  = int(n_ind_input) if n_ind_input else 20

    iter_input = input("\nMax optimisation iterations [default: 500]:\n> ").strip()
    max_iter   = int(iter_input) if iter_input else 500

    output_dir = input("\nFolder to save predictions.csv [default: data]:\n> ").strip().strip('"').strip("'")
    output_dir = output_dir if output_dir else "data"

    models, preds, normaliser, d_max = run_gpr(
        seed_path  = seed_path,
        pool_path  = pool_path,
        output_dir = output_dir,
        n_inducing = n_inducing,
        max_iter   = max_iter,
    )
   


MODULE 3 — HETEROSCEDASTIC GP MODEL



Path to seed_set.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\seed_set.csv

Path to pool_set.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\pool_set.csv

Number of inducing points [default: 20]:
>  

Max optimisation iterations [default: 500]:
>  

Folder to save predictions.csv [default: data]:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\training-data\predictions


STEP 1: Loading seed and pool sets
  Seed set : 24 points
  Pool set : 942 points

STEP 2: Computing composition space diameter (d_max)
  d_max (max composition distance): 116.7728 at.% units

STEP 2: Normalising inputs
  Input mean : [32.17666667 34.63041667]
  Input std  : [20.07049238 21.44890334]

STEP 3: Training heteroscedastic GP models

  ── Training model for: log_k0 ──────────────────
     Training points : 24
     Inducing points : 20
     Max iterations  : 500
    ELBO before training: -2374.6701
    ELBO after training:  -1.1685
     ELBO improvement: 2373.5016

  ── Training model for: alpha ──────────────────
     Training points : 24
     Inducing points : 20
     Max iterations  : 500
    ELBO before training: -118.8432
    ELBO after training:  60.3092
     ELBO improvement: 179.1524

DONE
  Predictions saved to : C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\training-data\predictions\predictions.csv
  log_k0 var range     : 0.0068 

In [13]:
print("\nTop 5 points the GP is most uncertain about (by log_k0 variance):")
top5 = preds.nlargest(5, 'log_k0_var')[
    ['library', 'area', 'Au [at.%]', 'Ir [at.%]', 'log_k0_var', 'alpha_var']
]
print(top5.to_string(index=False))


Top 5 points the GP is most uncertain about (by log_k0 variance):
library  area  Au [at.%]  Ir [at.%]  log_k0_var  alpha_var
Au-rich   325      83.82       7.89    0.147990   0.000645
Au-rich   336      83.36       6.88    0.136063   0.000628
Au-rich   326      83.19       7.33    0.133145   0.000615
Au-rich   313      83.11       8.21    0.133122   0.000605
Au-rich   337      82.97       6.07    0.126798   0.000615
